# 02 UV Comparison — Gold QA(可视化专用)
baseline/TD packed UV + 相同 raw atlas budget 的 rebake 对比。
此处 packing/rebake 仅用于 QA, 不影响 dataset acceptance。
依赖 PARTUV_ROOT teacher checkout(gold_evaluator)。

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import matplotlib.pyplot as plt
from meshuv.data.dataset import CleanDataset
from meshuv.visualization import pipeline_views as V

DATASET = "processed/clean_v1"      # 相对 MESHUV_DATA_ROOT(env 可覆盖)
root = V.resolve(DATASET)
ds = CleanDataset(root, expose_diagnostics=True) if root else None
print("objects:", ds and len(ds))

In [ ]:
from meshuv.evaluation.gold_evaluator import compare_methods
import torch, numpy as np
UID = None; RANK = None; SEED = 0
i = V.pick(ds, uid=UID, rank=RANK, seed=SEED)
item = ds[i]
# 如有 sanity checkpoint, 同时评 Student
sf = None
ck_path = "../reports/student_v0_sanity.ckpt"
if os.path.exists(ck_path):
    from meshuv.model.student_v0 import StudentV0
    from meshuv.data.collate import collate
    ck = torch.load(ck_path, map_location="cpu")
    model = StudentV0(d=ck["d"]); model.load_state_dict(ck["state"]); model.eval()
    b = collate([item])
    with torch.no_grad():
        pred = model(torch.as_tensor(b["features"]), b["object_ranges"],
                     torch.as_tensor(b["valid"])).numpy()
    A3 = item["targets"]["chart_surface_area"].astype(float)
    ash = A3 / A3.sum()
    sf = ash * np.exp(2 * pred)          # 线性 log 比 -> 面积份额(×2)
    sf = sf / sf.sum()
r = compare_methods(root, item, student_fraction=sf)
if r["status"] == "OK":
    names = [n for n in ("uniform", "teacher", "student") if f"{n}_uv" in r["draw"]]
    fig = plt.figure(figsize=(5.2 * len(names), 9))
    for k, n in enumerate(names):
        r["draw"][f"{n}_uv"](fig.add_subplot(2, len(names), k + 1))
        r["draw"][f"{n}_tex"](fig.add_subplot(2, len(names), len(names) + k + 1))
    plt.tight_layout(); plt.show()
    print(r["metrics_text"])
else:
    print("QA packing 失败:", r)